# Example 15: Hyperparameter Optimization with Ray Tune

Manually tuning hyperparameters -- learning rate, hidden size, model type --
is tedious and error-prone. TSFast integrates with
[Ray Tune](https://docs.ray.io/en/latest/tune/index.html) to automate the
search. This example runs a small hyperparameter search to find the best
model configuration for the Silverbox benchmark.

## Prerequisites

This example builds on concepts from:

- **Example 00** -- data loading and model training basics
- **Example 04** -- model architectures and `rnn_type`

Make sure Ray Tune is installed:

```bash
uv sync --extra dev
```

## Setup

Ray's uv integration requires the working directory to contain the project's
`pyproject.toml` when Ray workers are launched. Notebook kernels start in the
notebook's directory, so we move to the project root first.

In [1]:
import os
from pathlib import Path

_root = Path.cwd()
while not (_root / "pyproject.toml").is_file() and _root != _root.parent:
    _root = _root.parent
if (_root / "pyproject.toml").is_file():
    os.chdir(_root)

In [2]:
import identibench as idb

from tsfast.tsdata.benchmark import create_dls_silverbox
from tsfast.tune import LearnerTrainable, ray_device, report_metrics, resume_checkpoint
from tsfast.training import RNNLearner, fun_rmse
from ray import tune
from ray.tune.schedulers import PopulationBasedTraining

/home/dom_weber/development/tsfast/.claude/worktrees/fix-example-notebooks/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-02 19:23:09,844	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-07-02 19:23:10,061	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Why Hyperparameter Optimization?

Model performance depends heavily on hyperparameters: learning rate, hidden
size, architecture choice, and regularization strength. Finding the right
combination by hand requires many experiments and careful record-keeping.

Automated approaches help:

- **Grid search** evaluates every combination -- thorough but expensive.
- **Random search** samples randomly and is surprisingly effective in
  high-dimensional spaces.
- **Population-based training** evolves configurations during training,
  combining exploration with exploitation.

Ray Tune provides all of these strategies (and more) behind a unified API.
TSFast provides helpers that bridge a Learner and Ray Tune:

- **`ray_device()`** -- detects the GPU assigned to the current Ray worker.
- **`report_metrics(lrn)`** -- patches `log_epoch` to report metrics and
  checkpoints to Ray Tune after every epoch.
- **`resume_checkpoint(lrn)`** -- restores from a Ray Tune checkpoint when
  resuming a trial (e.g. population-based training).
- **`LearnerTrainable`** -- a class-based Trainable that wraps a Learner for
  schedulers like Population-Based Training that need checkpoint/restore and
  actor reuse.

## Prepare the DataLoaders

We use the Silverbox benchmark with a small batch size and window size to
keep the example lightweight. The benchmark spec defines an initialization
window that its evaluation protocol discards; using it as `n_skip` excludes
the same initial transient from the training loss.

In [3]:
dls = create_dls_silverbox(bs=16, win_sz=500, stp_sz=10)
n_skip = idb.BenchmarkSilverbox_Simulation.task.init_window

## Define a Training Function

Ray Tune calls a training function once per trial, each time with a different
hyperparameter configuration sampled from the search space. The DataLoaders
are captured via closure.

In [4]:
def train(config):
    lrn = RNNLearner(
        dls,
        rnn_type=config["rnn_type"],
        hidden_size=config["hidden_size"],
        n_skip=n_skip,
        metrics=[fun_rmse],
        device=ray_device(),
    )
    resume_checkpoint(lrn)
    with lrn.no_bar(), report_metrics(lrn):
        lrn.fit_flat_cos(config["n_epoch"], lr=config.get("lr"))

## Define the Search Space

The search space is a plain dictionary where values are Ray Tune sampling
primitives:

- **`tune.choice`** -- samples uniformly from a list of discrete options.
  Good for categorical parameters like architecture type or layer count.
- **`tune.loguniform`** -- samples uniformly on a logarithmic scale. Ideal
  for parameters that span orders of magnitude, such as learning rate.

Training parameters (`n_epoch`, `lr`) go in the same dict --
they are read by the training function and logged by Ray Tune.

## Run the Optimization

Pass the training function to `tune.Tuner` together with the search space.
`num_samples=4` runs 4 independent trials, each with a different
hyperparameter combination.

In [5]:
tuner = tune.Tuner(
    tune.with_resources(train, {"cpu": 1, "gpu": 0}),
    param_space={
        "rnn_type": tune.choice(["gru", "lstm"]),
        "hidden_size": tune.choice([32, 40]),
        "n_epoch": 3,
        "lr": 3e-3,
    },
    tune_config=tune.TuneConfig(metric="valid_loss", mode="min", num_samples=4),
)

results = tuner.fit()

(train pid=989622) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00000_0_hidden_size=40,rnn_type=lstm_2026-07-02_19-23-12/checkpoint_000000)


(raylet) [2026-07-02 19:23:21,071 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.104 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(train pid=989623) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00003_3_hidden_size=40,rnn_type=gru_2026-07-02_19-23-12/checkpoint_000000) [repeated 6x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(raylet) [2026-07-02 19:23:31,076 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.102 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.
(train pid=989621) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00002_2_hidden_size=40,rnn_type=gru_2026-07-02_19-23-12/checkpoint_000000)


(train pid=989623) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00003_3_hidden_size=40,rnn_type=gru_2026-07-02_19-23-12/checkpoint_000001)


(raylet) [2026-07-02 19:23:41,081 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.102 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(train pid=989621) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00002_2_hidden_size=40,rnn_type=gru_2026-07-02_19-23-12/checkpoint_000002) [repeated 2x across cluster]
2026-07-02 19:23:46,555	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/dom_weber/ray_results/train_2026-07-02_19-23-10' in 0.0021s.


2026-07-02 19:23:46,557	INFO tune.py:1039 -- Total run time: 33.98 seconds (33.96 seconds for the tuning loop).


## Analyze Results

`tuner.fit()` returns a `ResultGrid`. You can query it for the best trial
configuration, inspect per-trial metrics, or export data for further
analysis.

In [6]:
best = results.get_best_result()
print("Best config:")
for key in ["rnn_type", "hidden_size", "lr"]:
    print(f"  {key}: {best.config[key]}")

Best config:
  rnn_type: gru
  hidden_size: 40
  lr: 0.003


In [7]:
result_df = results.get_dataframe()
print("\nAll trial results:")
result_df[["config/rnn_type", "config/hidden_size", "valid_loss"]]


All trial results:


,config/rnn_type,config/hidden_size,valid_loss
0,lstm,40,0.000952
1,lstm,40,0.001012
2,gru,40,0.000897
3,gru,40,0.000981


## Using tune.loguniform for Learning Rate

In the first search we fixed the learning rate. A more thorough search treats
`lr` as a tunable parameter using `tune.loguniform`. This samples on a
logarithmic scale between the given bounds -- appropriate because the
difference between `1e-4` and `1e-3` matters more than between `1e-2` and
`1.1e-2`.

In [8]:
tuner_v2 = tune.Tuner(
    tune.with_resources(train, {"cpu": 1, "gpu": 0}),
    param_space={
        "rnn_type": tune.choice(["gru", "lstm"]),
        "hidden_size": tune.choice([32, 40]),
        "lr": tune.loguniform(1e-4, 1e-2),
        "n_epoch": 3,
    },
    tune_config=tune.TuneConfig(metric="valid_loss", mode="min", num_samples=4),
)

results_v2 = tuner_v2.fit()

(train pid=989623) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-10/train_b407c_00003_3_hidden_size=40,rnn_type=gru_2026-07-02_19-23-12/checkpoint_000002)


(raylet) [2026-07-02 19:23:51,085 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.101 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(train pid=991610) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-46/train_c84c1_00003_3_hidden_size=32,lr=0.0006,rnn_type=lstm_2026-07-02_19-23-46/checkpoint_000001) [repeated 3x across cluster]


(train pid=991608) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-46/train_c84c1_00002_2_hidden_size=32,lr=0.0008,rnn_type=gru_2026-07-02_19-23-46/checkpoint_000000) [repeated 4x across cluster]


(raylet) [2026-07-02 19:24:01,090 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.101 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(train pid=991608) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-46/train_c84c1_00002_2_hidden_size=32,lr=0.0008,rnn_type=gru_2026-07-02_19-23-46/checkpoint_000001) [repeated 2x across cluster]


(raylet) [2026-07-02 19:24:11,094 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.1 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(train pid=991608) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/train_2026-07-02_19-23-46/train_c84c1_00002_2_hidden_size=32,lr=0.0008,rnn_type=gru_2026-07-02_19-23-46/checkpoint_000002) [repeated 2x across cluster]


2026-07-02 19:24:20,067	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/dom_weber/ray_results/train_2026-07-02_19-23-46' in 0.0022s.


2026-07-02 19:24:20,070	INFO tune.py:1039 -- Total run time: 33.49 seconds (33.48 seconds for the tuning loop).


In [9]:
best_v2 = results_v2.get_best_result()
print("Best config (with lr search):")
for key in ["rnn_type", "hidden_size", "lr"]:
    print(f"  {key}: {best_v2.config[key]}")

Best config (with lr search):
  rnn_type: lstm
  hidden_size: 32
  lr: 0.0005854556771325755


## Population-Based Training

The examples above run each trial independently. **Population-Based Training
(PBT)** takes a different approach: it trains a population of models in
parallel, periodically copying weights from the best performers and perturbing
their hyperparameters. This combines the exploration of random search with the
exploitation of hand-tuning.

PBT requires the class-based Trainable API so that Ray can checkpoint, restore,
and mutate trials. `LearnerTrainable` provides this -- you supply a
`create_learner` factory and an optional `apply_config` callback for actor
reuse.

In [10]:
def create_learner(config):
    return RNNLearner(
        dls,
        rnn_type="gru",
        hidden_size=32,
        n_skip=n_skip,
        metrics=[fun_rmse],
        device=ray_device(),
    )


def apply_config(lrn, config):
    for pg in lrn.opt.param_groups:
        pg["lr"] = config["lr"]

In [11]:
pbt_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    perturbation_interval=2,
    hyperparam_mutations={"lr": tune.loguniform(1e-4, 1e-2)},
)

tuner_pbt = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(
            LearnerTrainable,
            create_learner=create_learner,
            apply_config=apply_config,
        ),
        {"cpu": 1, "gpu": 0},
    ),
    param_space={"lr": 3e-3},
    tune_config=tune.TuneConfig(
        metric="valid_loss",
        mode="min",
        num_samples=4,
        scheduler=pbt_scheduler,
    ),
    run_config=tune.RunConfig(stop={"training_iteration": 6}),
)

results_pbt = tuner_pbt.fit()

(raylet) [2026-07-02 19:24:21,099 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.1 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(raylet) [2026-07-02 19:24:31,104 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.1 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(raylet) [2026-07-02 19:24:41,109 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.1 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


2026-07-02 19:24:42,686	INFO pbt.py:741 -- [pbt]: no checkpoint for trial LearnerTrainable_dc457_00000. Skip exploit for Trial LearnerTrainable_dc457_00001


(LearnerTrainable pid=993370) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/LearnerTrainable_2026-07-02_19-24-20/LearnerTrainable_dc457_00002_2_2026-07-02_19-24-20/checkpoint_000000)


...

(raylet) [2026-07-02 19:25:31,140 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.099 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


(raylet) [2026-07-02 19:25:41,144 E 987720 987745] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-07-02_19-23-10_346738_987342 is over 95% full, available space: 101.099 GB; capacity: 3665.96 GB. Object creation will fail if spilling is required.


2026-07-02 19:25:44,700	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/dom_weber/ray_results/LearnerTrainable_2026-07-02_19-24-20' in 0.0017s.


(LearnerTrainable pid=995604) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/dom_weber/ray_results/LearnerTrainable_2026-07-02_19-24-20/LearnerTrainable_dc457_00000_0_2026-07-02_19-24-20/checkpoint_000000)


2026-07-02 19:25:44,703	INFO tune.py:1039 -- Total run time: 84.61 seconds (84.60 seconds for the tuning loop).


In [12]:
best_pbt = results_pbt.get_best_result()
print("Best PBT config:")
print(f"  lr: {best_pbt.config['lr']}")
print(f"  valid_loss: {best_pbt.metrics['valid_loss']:.4f}")

Best PBT config:
  lr: 0.003
  valid_loss: 0.0022


## Key Takeaways

- **`ray_device()`** detects the GPU assigned to the current Ray worker --
  pass it as `device` when constructing your Learner.
- **`report_metrics(lrn)`** patches `log_epoch` to report metrics and
  checkpoints to Ray Tune after every epoch.
- **`resume_checkpoint(lrn)`** restores from a Ray Tune checkpoint when
  resuming a trial (needed for population-based training).
- **`LearnerTrainable`** wraps a Learner as a class-based Trainable for
  schedulers like PBT that need checkpoint/restore and actor reuse. Supply a
  `create_learner` factory and an optional `apply_config` callback via
  `tune.with_parameters`.
- **`tune.choice`** for categorical parameters (architecture, layer count);
  **`tune.loguniform`** for continuous parameters on a log scale (learning
  rate). No custom samplers needed.
- **`tune.Tuner`** gives you full control over scheduling, stopping
  criteria, and resource allocation.
- **Start small** -- few trials, few epochs -- to validate the pipeline
  before scaling up.